In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv("data/processed/tsla_with_confidence.csv", index_col='Date', parse_dates=True)
print("Final dataset loaded with confidence and predictions")

In [ ]:
def compute_performance_metrics(df, strategy_col='equity_rule', initial_capital=100000):
    equity = df[strategy_col]
    returns = equity.pct_change().dropna()
    
    total_return = (equity.iloc[-1] / initial_capital - 1) * 100
    ann_return = returns.mean() * 252 * 100
    ann_vol = returns.std() * np.sqrt(252) * 100
    sharpe = returns.mean() / returns.std() * np.sqrt(252)
    max_dd = (equity / equity.cummax() - 1).min() * 100
    calmar = total_return / abs(max_dd) if max_dd != 0 else 0
    win_rate = (returns > 0).mean() * 100
    
    metrics = {
        "Total Return (%)": round(total_return, 2),
        "Annualized Return (%)": round(ann_return, 2),
        "Annualized Volatility (%)": round(ann_vol, 2),
        "Sharpe Ratio": round(sharpe, 3),
        "Max Drawdown (%)": round(max_dd, 2),
        "Calmar Ratio": round(calmar, 3),
        "Win Rate (%)": round(win_rate, 2),
        "Number of Trades": len(df[df['position'].abs() > 0.01]) if 'position' in df.columns else "N/A"
    }
    return pd.Series(metrics)

# Compare all strategies
print("=== PERFORMANCE SUMMARY ===")
bh_metrics = compute_performance_metrics(df, 'buy_hold')
rule_metrics = compute_performance_metrics(df, 'equity_rule')

comparison = pd.DataFrame({
    "Buy & Hold": bh_metrics,
    "XAI-RL Strategy": rule_metrics
})

print(comparison.round(3))
comparison.to_csv("results/performance_comparison.csv")

In [ ]:
plt.figure(figsize=(15, 10))

# Equity Curve
plt.subplot(2, 2, 1)
plt.plot(df.index, df['buy_hold'], label='Buy & Hold', alpha=0.8)
plt.plot(df.index, df['equity_rule'], label='XAI-RL Hybrid', linewidth=2)
plt.title('Equity Curve')
plt.legend()
plt.grid(True)

# Drawdown
plt.subplot(2, 2, 2)
drawdown = df['equity_rule'] / df['equity_rule'].cummax() - 1
plt.plot(df.index, drawdown * 100, label='Strategy Drawdown', color='red')
plt.title('Drawdown Curve')
plt.ylabel('%')
plt.legend()
plt.grid(True)

# Confidence vs Returns
plt.subplot(2, 2, 3)
plt.scatter(df['confidence_score'], df['returns'], alpha=0.6)
plt.title('Confidence Score vs Daily Returns')
plt.xlabel('Confidence')
plt.ylabel('Daily Return')
plt.grid(True)

plt.tight_layout()
plt.savefig("results/final_analysis_plots.png", dpi=200)
plt.show()

In [ ]:
summary = f"""
XAI-RL Framework Replication Results (TSLA 2021-2022)

Best Parameters:
- Confidence Threshold : {best_params.get('threshold', 'N/A')}
- Base Position Size   : {best_params.get('base_position', 'N/A')}

Key Metrics:
- Total Return (Strategy): {rule_metrics['Total Return (%)']}%
- Sharpe Ratio           : {rule_metrics['Sharpe Ratio']}
- Max Drawdown           : {rule_metrics['Max Drawdown (%)']}%
- Calmar Ratio           : {rule_metrics['Calmar Ratio']}

Comparison vs Buy & Hold: {'Outperformed' if rule_metrics['Sharpe Ratio'] > bh_metrics['Sharpe Ratio'] else 'Underperformed'}
"""

with open("results/replication_summary.txt", "w") as f:
    f.write(summary)

print(summary)